# XGBoost: 預測用餐時段 (Reservation Hour)

目標：以 `reservation_hour`（0-23 的整數小時）為預測目標，使用 XGBoost Classifier 進行多類別分類。

**Preprocessing 流程**：`features_ready.csv` → Drop NA → Drop leakage 欄位 → Train/Test Split → Label Encoding（不做 Scaling）

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from src.exploration.preprocessing.common import load_features, split_data, fill_missing
from src.exploration.preprocessing.tree import preprocess_for_tree

sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# --- Data Loading & Overview ---
df = load_features("../data/processed/features_ready.csv")

print(f"Shape: {df.shape}")
print(f"\n--- Missing Values ---")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\n--- Data Types ---")
print(df.dtypes)
df.head(3)

In [ ]:
# --- Drop NA ---
df = df.dropna()
print(f"Shape after dropna: {df.shape}")

In [ ]:
# --- Target Distribution ---
fig, ax = plt.subplots(figsize=(12, 4))

df["reservation_hour"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Reservation Hour Distribution")
ax.set_xlabel("Hour (0-23)")
ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

print(df["reservation_hour"].describe())
print(f"\nUnique hours: {sorted(df['reservation_hour'].unique())}")

In [ ]:
# --- Preprocessing ---
TARGET = "reservation_hour"
LEAKAGE_COLS = ["reservation_seconds", "reservation_half_hour"]  # derived from same datetime as target

df_clean = df.drop(columns=LEAKAGE_COLS)

X_train, X_test, y_train, y_test = split_data(df_clean, target_col=TARGET)
X_train_proc, X_test_proc, encoders = preprocess_for_tree(X_train, X_test)

# Encode target to contiguous integers (XGBoost requires 0..n_classes-1)
from sklearn.preprocessing import LabelEncoder
target_encoder = LabelEncoder()
y_train = pd.Series(target_encoder.fit_transform(y_train), index=y_train.index)
y_test = pd.Series(target_encoder.transform(y_test), index=y_test.index)

print(f"Dropped leakage columns: {LEAKAGE_COLS}")
print(f"Train: {X_train_proc.shape}, Test: {X_test_proc.shape}")
print(f"Label-encoded columns: {list(encoders.keys())}")
print(f"Target classes: {sorted(y_train.unique())} -> hours: {list(target_encoder.classes_)}")

In [ ]:
# --- Model Training ---
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
)
model.fit(X_train_proc, y_train)

y_pred = model.predict(X_test_proc)
print("Model training completed.")
print(f"Trees: {model.n_estimators}, Max depth: {model.max_depth}, LR: {model.learning_rate}")

In [ ]:
# --- Evaluation Metrics ---
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print("=== XGBoost Performance ===")
print(f"Accuracy : {acc:.4f}")
print(f"F1 (weighted): {f1:.4f}")
print(f"\n--- Classification Report ---")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# --- Confusion Matrix ---
labels = sorted(y_test.unique())
hour_labels = target_encoder.inverse_transform(labels)
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap (counts)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=hour_labels, yticklabels=hour_labels, ax=axes[0])
axes[0].set_xlabel("Predicted Hour")
axes[0].set_ylabel("Actual Hour")
axes[0].set_title("Confusion Matrix (Counts)")

# Heatmap (normalized by row)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=hour_labels, yticklabels=hour_labels, ax=axes[1])
axes[1].set_xlabel("Predicted Hour")
axes[1].set_ylabel("Actual Hour")
axes[1].set_title("Confusion Matrix (Row-Normalized)")

plt.tight_layout()
plt.show()

In [ ]:
# --- Per-Class Accuracy & Prediction Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-class accuracy (display with original hour labels)
per_class = pd.DataFrame({"actual": y_test, "pred": y_pred})
class_acc = per_class.groupby("actual").apply(lambda g: (g["actual"] == g["pred"]).mean())
class_acc.index = target_encoder.inverse_transform(class_acc.index)
class_acc.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Per-Class Accuracy")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Accuracy")
axes[0].axhline(acc, color="red", linestyle="--", label=f"Overall: {acc:.3f}")
axes[0].legend()

# Distribution comparison (map back to original hours)
y_test_hours = target_encoder.inverse_transform(y_test)
y_pred_hours = target_encoder.inverse_transform(y_pred)
axes[1].hist(y_test_hours, bins=range(0, 25), alpha=0.5, label="Actual", color="steelblue", edgecolor="white", align="left")
axes[1].hist(y_pred_hours, bins=range(0, 25), alpha=0.5, label="Predicted", color="coral", edgecolor="white", align="left")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution: Actual vs Predicted")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importance (Top 20) ---
importance_df = pd.DataFrame({
    "feature": X_train_proc.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

top_n = 20
top_features = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_features["importance"], color="steelblue")
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features["feature"])
ax.invert_yaxis()
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title(f"Top {top_n} Feature Importance (XGBoost)")
plt.tight_layout()
plt.show()

print(f"\n--- All feature importances ---")
display(importance_df.reset_index(drop=True))

## Summary

| Metric | Description |
|--------|-------------|
| Accuracy | 預測正確的比例 |
| F1 (weighted) | 加權 F1 Score，考慮各類別樣本數 |
| Confusion Matrix | 各小時的預測 vs 實際分布 |
| Feature Importance | Gain，衡量每個特徵在所有樹中被用於分裂時帶來的平均增益 |

## Sub-Dataset Analysis: 午餐 / 下午茶 / 晚餐

將資料依 `reservation_hour` 拆分為三個子資料集，分別建立獨立的 XGBoost 模型，重複相同的分析流程。

| Subset | reservation_hour |
|--------|------------------|
| 午餐 Lunch | 11 - 14 |
| 下午茶 Afternoon Tea | 15 - 17 |
| 晚餐 Dinner | 18 - 23 |

In [ ]:
# --- Split data by meal time ---
df_lunch = df_clean[df_clean["reservation_hour"].between(11, 14)]
df_afternoon = df_clean[df_clean["reservation_hour"].between(15, 17)]
df_dinner = df_clean[df_clean["reservation_hour"].between(18, 23)]

print(f"午餐 Lunch      (11-14h): {len(df_lunch):>6,} rows")
print(f"下午茶 Afternoon (15-17h): {len(df_afternoon):>6,} rows")
print(f"晚餐 Dinner     (18-23h): {len(df_dinner):>6,} rows")


In [ ]:
def run_subset_analysis(df_subset, label):
    """對子資料集執行完整的 XGBoost Classification 分析。"""
    TARGET = "reservation_hour"
    LEAKAGE_COLS = ["reservation_seconds", "reservation_half_hour"]

    df_clean = df_subset.drop(columns=LEAKAGE_COLS, errors="ignore")
    X_tr, X_te, y_tr, y_te = split_data(df_clean, target_col=TARGET)
    X_tr_proc, X_te_proc, _ = preprocess_for_tree(X_tr, X_te)

    # Encode target to contiguous integers
    sub_target_enc = LabelEncoder()
    y_tr = pd.Series(sub_target_enc.fit_transform(y_tr), index=y_tr.index)
    y_te = pd.Series(sub_target_enc.transform(y_te), index=y_te.index)

    # Train
    mdl = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, eval_metric="mlogloss",
    )
    mdl.fit(X_tr_proc, y_tr)
    y_pr = mdl.predict(X_te_proc)

    # Metrics
    sub_acc = accuracy_score(y_te, y_pr)
    sub_f1  = f1_score(y_te, y_pr, average="weighted", zero_division=0)

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Samples : {len(df_subset):,}  (Train {len(X_tr):,} / Test {len(X_te):,})")
    print(f"  Features: {X_tr_proc.shape[1]}")
    print(f"  Accuracy    : {sub_acc:.4f}")
    print(f"  F1 (weighted): {sub_f1:.4f}")

    # Map back to original hours for display
    hour_labels = sub_target_enc.inverse_transform(sorted(y_te.unique()))

    # --- 2x3 plot grid ---
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"{label} — XGBoost Classifier", fontsize=14, fontweight="bold")

    # (0,0) Confusion matrix
    sub_labels = sorted(y_te.unique())
    cm = confusion_matrix(y_te, y_pr, labels=sub_labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=hour_labels, yticklabels=hour_labels, ax=axes[0, 0])
    axes[0, 0].set_xlabel("Predicted")
    axes[0, 0].set_ylabel("Actual")
    axes[0, 0].set_title("Confusion Matrix")

    # (0,1) Normalized confusion matrix
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=hour_labels, yticklabels=hour_labels, ax=axes[0, 1])
    axes[0, 1].set_xlabel("Predicted")
    axes[0, 1].set_ylabel("Actual")
    axes[0, 1].set_title("Confusion Matrix (Normalized)")

    # (0,2) Target distribution
    df_subset["reservation_hour"].value_counts().sort_index().plot(
        kind="bar", ax=axes[0, 2], color="steelblue")
    axes[0, 2].set_title("Target Distribution")
    axes[0, 2].set_xlabel("Hour")
    axes[0, 2].set_ylabel("Count")

    # (1,0) Per-class accuracy
    pc = pd.DataFrame({"actual": y_te, "pred": y_pr})
    ca = pc.groupby("actual").apply(lambda g: (g["actual"] == g["pred"]).mean())
    ca.index = sub_target_enc.inverse_transform(ca.index)
    ca.plot(kind="bar", ax=axes[1, 0], color="steelblue")
    axes[1, 0].axhline(sub_acc, color="red", linestyle="--", label=f"Overall: {sub_acc:.3f}")
    axes[1, 0].set_title("Per-Class Accuracy")
    axes[1, 0].set_xlabel("Hour")
    axes[1, 0].set_ylabel("Accuracy")
    axes[1, 0].legend()

    # (1,1) Distribution comparison (original hours)
    y_te_hours = sub_target_enc.inverse_transform(y_te)
    y_pr_hours = sub_target_enc.inverse_transform(y_pr)
    axes[1, 1].hist(y_te_hours, bins=range(int(y_te_hours.min()), int(y_te_hours.max())+2), alpha=0.5,
                    label="Actual", color="steelblue", edgecolor="white", align="left")
    axes[1, 1].hist(y_pr_hours, bins=range(int(y_pr_hours.min()), int(y_pr_hours.max())+2), alpha=0.5,
                    label="Predicted", color="coral", edgecolor="white", align="left")
    axes[1, 1].set_xlabel("Hour")
    axes[1, 1].set_title("Actual vs Predicted Distribution")
    axes[1, 1].legend()

    # (1,2) Feature importance top 15
    imp_df = pd.DataFrame({
        "feature": X_tr_proc.columns,
        "importance": mdl.feature_importances_
    }).sort_values("importance", ascending=False)
    top = imp_df.head(15)
    axes[1, 2].barh(range(len(top)), top["importance"], color="steelblue")
    axes[1, 2].set_yticks(range(len(top)))
    axes[1, 2].set_yticklabels(top["feature"], fontsize=8)
    axes[1, 2].invert_yaxis()
    axes[1, 2].set_xlabel("Feature Importance (Gain)")
    axes[1, 2].set_title("Top 15 Features")

    plt.tight_layout()
    plt.show()

    return {"label": label, "n": len(df_subset), "accuracy": sub_acc, "f1_weighted": sub_f1}

print("run_subset_analysis() defined.")

In [ ]:
# --- Run analysis on each subset ---
results = []
for subset, label in [
    (df_lunch,     "午餐 Lunch (11-14h)"),
    (df_afternoon, "下午茶 Afternoon Tea (15-17h)"),
    (df_dinner,    "晚餐 Dinner (18-23h)"),
]:
    results.append(run_subset_analysis(subset, label))

In [ ]:
# ============================================================
#  Comparison Summary
# ============================================================
print("="*70)
print("  COMPARISON SUMMARY (vs Full Dataset)")
print("="*70)

full_result = {
    "label": "全部 Full Dataset",
    "n": len(df),
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
}
all_results = [full_result] + results

comp_df = pd.DataFrame(all_results)
display(comp_df[["label", "n", "accuracy", "f1_weighted"]].rename(columns={
    "label": "Subset", "n": "Samples", "accuracy": "Accuracy", "f1_weighted": "F1 (weighted)"
}).to_string(index=False))

# Comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels_short = [r["label"] for r in all_results]
colors = ["gray", "steelblue", "coral", "seagreen"]
x = range(len(labels_short))

for ax, col, title in [
    (axes[0], "accuracy",    "Accuracy"),
    (axes[1], "f1_weighted", "F1 (weighted)"),
]:
    ax.bar(x, comp_df[col], color=colors)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels_short, fontsize=9, rotation=15, ha="right")
    ax.set_title(title)
    ax.set_ylim(0, 1)

plt.suptitle("Full vs Subset Comparison (XGBoost Classifier)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Sub-Dataset Analysis (二分法): 午餐 / 晚餐

另一種切割方式：將資料依 `reservation_hour` 拆分為兩個子資料集。

| Subset | reservation_hour |
|--------|------------------|
| 午餐 Lunch2 | 9 - 14 |
| 晚餐 Dinner2 | 15 - 23 |

In [ ]:
# --- Split data (二分法) ---
df_lunch2 = df[df["reservation_hour"].between(9, 14)]
df_dinner2 = df[df["reservation_hour"].between(15, 23)]

print(f"午餐 Lunch2   ( 9-14h): {len(df_lunch2):>6,} rows")
print(f"晚餐 Dinner2  (15-23h): {len(df_dinner2):>6,} rows")
print(f"其他 Other    ( 0- 8h): {len(df) - len(df_lunch2) - len(df_dinner2):>6,} rows")
print(f"{'─'*42}")
print(f"Total                 : {len(df):>6,} rows")

In [ ]:
# --- Run analysis (二分法) ---
results2 = []
for subset, label in [
    (df_lunch2,  "午餐 Lunch2 (9-14h)"),
    (df_dinner2, "晚餐 Dinner2 (15-23h)"),
]:
    results2.append(run_subset_analysis(subset, label))

In [ ]:
# ============================================================
#  Comparison Summary (二分法 vs Full Dataset)
# ============================================================
print("="*70)
print("  COMPARISON SUMMARY — 二分法 (vs Full Dataset)")
print("="*70)

full_result2 = {
    "label": "全部 Full Dataset",
    "n": len(df),
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
}
all_results2 = [full_result2] + results2

comp_df2 = pd.DataFrame(all_results2)
display(comp_df2[["label", "n", "accuracy", "f1_weighted"]].rename(columns={
    "label": "Subset", "n": "Samples", "accuracy": "Accuracy", "f1_weighted": "F1 (weighted)"
}).to_string(index=False))

# Comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels_short2 = [r["label"] for r in all_results2]
colors2 = ["gray", "steelblue", "coral"]
x2 = range(len(labels_short2))

for ax, col, title in [
    (axes[0], "accuracy",    "Accuracy"),
    (axes[1], "f1_weighted", "F1 (weighted)"),
]:
    ax.bar(x2, comp_df2[col], color=colors2)
    ax.set_xticks(list(x2))
    ax.set_xticklabels(labels_short2, fontsize=9, rotation=15, ha="right")
    ax.set_title(title)
    ax.set_ylim(0, 1)

plt.suptitle("Full vs Subset Comparison — 二分法 (XGBoost Classifier)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Half-Hour Granularity Analysis (半小時顆粒度)

將預測目標改為 `reservation_half_hour`（0-47 的半小時時段），以更細的時間解析度進行 XGBoost 分類。

**Target**: `reservation_half_hour` (0-47)
**Leakage**: `reservation_hour`, `reservation_seconds`（皆由同一 datetime 衍生）

In [ ]:
# --- Half-Hour: Data Reload ---
df_hh = load_features("../data/processed/features_ready.csv")
df_hh = df_hh.dropna()

# Target distribution
fig, ax = plt.subplots(figsize=(16, 4))
slot_counts = df_hh["reservation_half_hour"].value_counts().sort_index()
slot_labels = [f"{s // 2}:{(s % 2) * 30:02d}" for s in slot_counts.index]
ax.bar(range(len(slot_counts)), slot_counts.values, color="steelblue")
ax.set_xticks(range(len(slot_counts)))
ax.set_xticklabels(slot_labels, rotation=45, ha="right", fontsize=8)
ax.set_title("Reservation Half-Hour Slot Distribution")
ax.set_xlabel("Half-Hour Slot")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Shape: {df_hh.shape}")
print(f"Unique slots: {sorted(df_hh['reservation_half_hour'].unique())}")
print(f"Number of classes: {df_hh['reservation_half_hour'].nunique()}")

In [ ]:
# --- Half-Hour: Preprocessing & Training ---
TARGET_HH = "reservation_half_hour"
LEAKAGE_HH = ["reservation_hour", "reservation_seconds"]

df_hh_clean = df_hh.drop(columns=LEAKAGE_HH)
X_train_hh, X_test_hh, y_train_hh, y_test_hh = split_data(df_hh_clean, target_col=TARGET_HH)
X_train_hh_proc, X_test_hh_proc, encoders_hh = preprocess_for_tree(X_train_hh, X_test_hh)

# Encode target to contiguous integers
target_enc_hh = LabelEncoder()
y_train_hh = pd.Series(target_enc_hh.fit_transform(y_train_hh), index=y_train_hh.index)
y_test_hh = pd.Series(target_enc_hh.transform(y_test_hh), index=y_test_hh.index)

# Train
model_hh = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1, eval_metric="mlogloss",
)
model_hh.fit(X_train_hh_proc, y_train_hh)
y_pred_hh = model_hh.predict(X_test_hh_proc)

# Metrics
acc_hh = accuracy_score(y_test_hh, y_pred_hh)
f1_hh = f1_score(y_test_hh, y_pred_hh, average="weighted")

print("=== XGBoost Half-Hour Performance ===")
print(f"Accuracy : {acc_hh:.4f}")
print(f"F1 (weighted): {f1_hh:.4f}")
print(f"Classes: {len(target_enc_hh.classes_)}")
print(f"\n--- Classification Report ---")
print(classification_report(y_test_hh, y_pred_hh, zero_division=0))

In [ ]:
# --- Half-Hour: Plots ---
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("XGBoost — Half-Hour Granularity (Full Dataset)", fontsize=14, fontweight="bold")

# (0,0) Confusion matrix (normalized)
hh_labels = sorted(y_test_hh.unique())
hh_display = [f"{s // 2}:{(s % 2) * 30:02d}" for s in target_enc_hh.inverse_transform(hh_labels)]
cm_hh = confusion_matrix(y_test_hh, y_pred_hh, labels=hh_labels)
cm_hh_norm = cm_hh.astype(float) / cm_hh.sum(axis=1, keepdims=True)
sns.heatmap(cm_hh_norm, annot=False, cmap="Blues",
            xticklabels=hh_display, yticklabels=hh_display, ax=axes[0, 0])
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Actual")
axes[0, 0].set_title("Confusion Matrix (Normalized)")
axes[0, 0].tick_params(axis='both', labelsize=6)

# (0,1) Per-class accuracy
pc_hh = pd.DataFrame({"actual": y_test_hh, "pred": y_pred_hh})
ca_hh = pc_hh.groupby("actual").apply(lambda g: (g["actual"] == g["pred"]).mean())
ca_hh.index = [f"{s // 2}:{(s % 2) * 30:02d}" for s in target_enc_hh.inverse_transform(ca_hh.index)]
ca_hh.plot(kind="bar", ax=axes[0, 1], color="steelblue")
axes[0, 1].axhline(acc_hh, color="red", linestyle="--", label=f"Overall: {acc_hh:.3f}")
axes[0, 1].set_title("Per-Class Accuracy")
axes[0, 1].set_xlabel("Half-Hour Slot")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].tick_params(axis='x', labelsize=6)
axes[0, 1].legend()

# (1,0) Distribution comparison
y_test_hh_slots = target_enc_hh.inverse_transform(y_test_hh)
y_pred_hh_slots = target_enc_hh.inverse_transform(y_pred_hh)
axes[1, 0].hist(y_test_hh_slots, bins=range(0, 49), alpha=0.5, label="Actual", color="steelblue", edgecolor="white", align="left")
axes[1, 0].hist(y_pred_hh_slots, bins=range(0, 49), alpha=0.5, label="Predicted", color="coral", edgecolor="white", align="left")
axes[1, 0].set_xlabel("Half-Hour Slot")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Distribution: Actual vs Predicted")
axes[1, 0].legend()

# (1,1) Feature importance top 20
imp_hh = pd.DataFrame({
    "feature": X_train_hh_proc.columns,
    "importance": model_hh.feature_importances_
}).sort_values("importance", ascending=False)
top_hh = imp_hh.head(20)
axes[1, 1].barh(range(len(top_hh)), top_hh["importance"], color="steelblue")
axes[1, 1].set_yticks(range(len(top_hh)))
axes[1, 1].set_yticklabels(top_hh["feature"], fontsize=8)
axes[1, 1].invert_yaxis()
axes[1, 1].set_xlabel("Feature Importance (Gain)")
axes[1, 1].set_title("Top 20 Features")

plt.tight_layout()
plt.show()

In [ ]:
def run_subset_analysis_hh(df_subset, label):
    """對子資料集執行半小時顆粒度的 XGBoost Classification 分析。"""
    TARGET_HH = "reservation_half_hour"
    LEAKAGE_HH = ["reservation_hour", "reservation_seconds"]

    df_clean = df_subset.drop(columns=LEAKAGE_HH, errors="ignore")
    X_tr, X_te, y_tr, y_te = split_data(df_clean, target_col=TARGET_HH)
    X_tr_proc, X_te_proc, _ = preprocess_for_tree(X_tr, X_te)

    # Encode target to contiguous integers
    sub_enc = LabelEncoder()
    y_tr = pd.Series(sub_enc.fit_transform(y_tr), index=y_tr.index)
    y_te = pd.Series(sub_enc.transform(y_te), index=y_te.index)

    mdl = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, eval_metric="mlogloss",
    )
    mdl.fit(X_tr_proc, y_tr)
    y_pr = mdl.predict(X_te_proc)

    sub_acc = accuracy_score(y_te, y_pr)
    sub_f1  = f1_score(y_te, y_pr, average="weighted", zero_division=0)

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Samples : {len(df_subset):,}  (Train {len(X_tr):,} / Test {len(X_te):,})")
    print(f"  Classes : {len(sub_enc.classes_)}")
    print(f"  Accuracy    : {sub_acc:.4f}")
    print(f"  F1 (weighted): {sub_f1:.4f}")

    slot_labels = sub_enc.inverse_transform(sorted(y_te.unique()))
    hh_display = [f"{s // 2}:{(s % 2) * 30:02d}" for s in slot_labels]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"{label} — XGBoost (Half-Hour)", fontsize=14, fontweight="bold")

    # (0,0) Confusion matrix
    sub_labels = sorted(y_te.unique())
    cm = confusion_matrix(y_te, y_pr, labels=sub_labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=hh_display, yticklabels=hh_display, ax=axes[0, 0])
    axes[0, 0].set_xlabel("Predicted")
    axes[0, 0].set_ylabel("Actual")
    axes[0, 0].set_title("Confusion Matrix")
    axes[0, 0].tick_params(axis='both', labelsize=7)

    # (0,1) Normalized confusion matrix
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=hh_display, yticklabels=hh_display, ax=axes[0, 1])
    axes[0, 1].set_xlabel("Predicted")
    axes[0, 1].set_ylabel("Actual")
    axes[0, 1].set_title("Confusion Matrix (Normalized)")
    axes[0, 1].tick_params(axis='both', labelsize=7)

    # (0,2) Target distribution
    df_subset["reservation_half_hour"].value_counts().sort_index().plot(
        kind="bar", ax=axes[0, 2], color="steelblue")
    axes[0, 2].set_title("Target Distribution")
    axes[0, 2].set_xlabel("Half-Hour Slot")
    axes[0, 2].set_ylabel("Count")

    # (1,0) Per-class accuracy
    pc = pd.DataFrame({"actual": y_te, "pred": y_pr})
    ca = pc.groupby("actual").apply(lambda g: (g["actual"] == g["pred"]).mean())
    ca.index = [f"{s // 2}:{(s % 2) * 30:02d}" for s in sub_enc.inverse_transform(ca.index)]
    ca.plot(kind="bar", ax=axes[1, 0], color="steelblue")
    axes[1, 0].axhline(sub_acc, color="red", linestyle="--", label=f"Overall: {sub_acc:.3f}")
    axes[1, 0].set_title("Per-Class Accuracy")
    axes[1, 0].set_xlabel("Half-Hour Slot")
    axes[1, 0].tick_params(axis='x', labelsize=7)
    axes[1, 0].legend()

    # (1,1) Distribution comparison
    y_te_s = sub_enc.inverse_transform(y_te)
    y_pr_s = sub_enc.inverse_transform(y_pr)
    axes[1, 1].hist(y_te_s, bins=range(int(y_te_s.min()), int(y_te_s.max())+2), alpha=0.5,
                    label="Actual", color="steelblue", edgecolor="white", align="left")
    axes[1, 1].hist(y_pr_s, bins=range(int(y_pr_s.min()), int(y_pr_s.max())+2), alpha=0.5,
                    label="Predicted", color="coral", edgecolor="white", align="left")
    axes[1, 1].set_xlabel("Half-Hour Slot")
    axes[1, 1].set_title("Actual vs Predicted Distribution")
    axes[1, 1].legend()

    # (1,2) Feature importance top 15
    imp_df = pd.DataFrame({
        "feature": X_tr_proc.columns,
        "importance": mdl.feature_importances_
    }).sort_values("importance", ascending=False)
    top = imp_df.head(15)
    axes[1, 2].barh(range(len(top)), top["importance"], color="steelblue")
    axes[1, 2].set_yticks(range(len(top)))
    axes[1, 2].set_yticklabels(top["feature"], fontsize=8)
    axes[1, 2].invert_yaxis()
    axes[1, 2].set_xlabel("Feature Importance (Gain)")
    axes[1, 2].set_title("Top 15 Features")

    plt.tight_layout()
    plt.show()

    return {"label": label, "n": len(df_subset), "accuracy": sub_acc, "f1_weighted": sub_f1}

# --- Split & Run (三分法) ---
df_hh_lunch = df_hh[df_hh["reservation_hour"].between(11, 14)]
df_hh_afternoon = df_hh[df_hh["reservation_hour"].between(15, 17)]
df_hh_dinner = df_hh[df_hh["reservation_hour"].between(18, 23)]

print(f"午餐 Lunch      (11-14h): {len(df_hh_lunch):>6,} rows")
print(f"下午茶 Afternoon (15-17h): {len(df_hh_afternoon):>6,} rows")
print(f"晚餐 Dinner     (18-23h): {len(df_hh_dinner):>6,} rows")

results_hh = []
for subset, label in [
    (df_hh_lunch,     "午餐 Lunch (11-14h)"),
    (df_hh_afternoon, "下午茶 Afternoon Tea (14-18h)"),
    (df_hh_dinner,    "晚餐 Dinner (18-23h)"),
]:
    results_hh.append(run_subset_analysis_hh(subset, label))

In [ ]:
# ============================================================
#  Comparison Summary — Half-Hour (vs Full Dataset)
# ============================================================
print("="*70)
print("  COMPARISON SUMMARY — Half-Hour Granularity (vs Full Dataset)")
print("="*70)

full_result_hh = {
    "label": "全部 Full Dataset",
    "n": len(df_hh),
    "accuracy": acc_hh,
    "f1_weighted": f1_hh,
}
all_results_hh = [full_result_hh] + results_hh

comp_df_hh = pd.DataFrame(all_results_hh)
display(comp_df_hh[["label", "n", "accuracy", "f1_weighted"]].rename(columns={
    "label": "Subset", "n": "Samples", "accuracy": "Accuracy", "f1_weighted": "F1 (weighted)"
}).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels_hh = [r["label"] for r in all_results_hh]
colors_hh = ["gray", "steelblue", "coral", "seagreen"]
x_hh = range(len(labels_hh))

for ax, col, title in [
    (axes[0], "accuracy",    "Accuracy"),
    (axes[1], "f1_weighted", "F1 (weighted)"),
]:
    ax.bar(x_hh, comp_df_hh[col], color=colors_hh)
    ax.set_xticks(list(x_hh))
    ax.set_xticklabels(labels_hh, fontsize=9, rotation=15, ha="right")
    ax.set_title(title)
    ax.set_ylim(0, 1)

plt.suptitle("Full vs Subset — Half-Hour (XGBoost Classifier)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Reservation Seconds Regression (秒級預測)

將預測目標改為 `reservation_seconds`（0-86399 秒，午夜起算的秒數），使用 XGBoost **Regressor** 進行迴歸預測。

**Target**: `reservation_seconds` (0-86399)
**Leakage**: `reservation_hour`, `reservation_half_hour`（皆由同一 datetime 衍生）
**Metrics**: RMSE, MAE, R²

In [ ]:
# --- Seconds: Data Reload ---
df_sec = load_features("../data/processed/features_ready.csv")
df_sec = df_sec.dropna()

# Target distribution
fig, ax = plt.subplots(figsize=(14, 4))
ax.hist(df_sec["reservation_seconds"], bins=100, color="steelblue", edgecolor="white")
ax.set_title("Reservation Seconds Distribution")
ax.set_xlabel("Seconds since midnight")
ax.set_ylabel("Count")

# Add hour labels
sec_ticks = list(range(0, 86400, 3600))
sec_labels = [f"{s // 3600}:00" for s in sec_ticks]
ax.set_xticks(sec_ticks)
ax.set_xticklabels(sec_labels, rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Shape: {df_sec.shape}")
print(f"Target range: {df_sec['reservation_seconds'].min()} ~ {df_sec['reservation_seconds'].max()}")
print(f"Target mean: {df_sec['reservation_seconds'].mean():.0f} ({df_sec['reservation_seconds'].mean()/3600:.1f} hours)")

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Seconds: Preprocessing & Training ---
TARGET_SEC = "reservation_seconds"
LEAKAGE_SEC = ["reservation_hour", "reservation_half_hour"]

df_sec_clean = df_sec.drop(columns=LEAKAGE_SEC)
X_train_sec, X_test_sec, y_train_sec, y_test_sec = split_data(df_sec_clean, target_col=TARGET_SEC)
X_train_sec_proc, X_test_sec_proc, encoders_sec = preprocess_for_tree(X_train_sec, X_test_sec)

# Train
model_sec = XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1,
)
model_sec.fit(X_train_sec_proc, y_train_sec)
y_pred_sec = model_sec.predict(X_test_sec_proc)

# Metrics
rmse_sec = mean_squared_error(y_test_sec, y_pred_sec)
mae_sec = mean_absolute_error(y_test_sec, y_pred_sec)
r2_sec = r2_score(y_test_sec, y_pred_sec)

print("=== XGBoost Seconds-Level Regression ===")
print(f"RMSE : {rmse_sec:,.0f} seconds  ({rmse_sec/60:.1f} minutes)")
print(f"MAE  : {mae_sec:,.0f} seconds  ({mae_sec/60:.1f} minutes)")
print(f"R²   : {r2_sec:.4f}")
print(f"Train size: {len(X_train_sec):,}  |  Test size: {len(X_test_sec):,}")

In [ ]:
# --- Seconds: Plots ---
residuals_sec = y_test_sec - y_pred_sec

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle("XGBoost — Seconds-Level Regression (Full Dataset)", fontsize=14, fontweight="bold")

# (0,0) Actual vs Predicted scatter
axes[0, 0].scatter(y_test_sec, y_pred_sec, alpha=0.15, s=5, color="steelblue")
lims = [min(y_test_sec.min(), y_pred_sec.min()), max(y_test_sec.max(), y_pred_sec.max())]
axes[0, 0].plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
axes[0, 0].set_xlabel("Actual (seconds)")
axes[0, 0].set_ylabel("Predicted (seconds)")
axes[0, 0].set_title("Actual vs Predicted")
axes[0, 0].legend()

# (0,1) Residual distribution
axes[0, 1].hist(residuals_sec / 60, bins=80, color="steelblue", edgecolor="white")
axes[0, 1].axvline(0, color="red", linestyle="--")
axes[0, 1].set_xlabel("Residual (minutes)")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title(f"Residual Distribution (MAE={mae_sec/60:.1f} min)")

# (1,0) Residual vs Predicted
axes[1, 0].scatter(y_pred_sec, residuals_sec / 60, alpha=0.15, s=5, color="steelblue")
axes[1, 0].axhline(0, color="red", linestyle="--")
axes[1, 0].set_xlabel("Predicted (seconds)")
axes[1, 0].set_ylabel("Residual (minutes)")
axes[1, 0].set_title("Residual vs Predicted")

# (1,1) Feature importance top 20
imp_sec = pd.DataFrame({
    "feature": X_train_sec_proc.columns,
    "importance": model_sec.feature_importances_
}).sort_values("importance", ascending=False)
top_sec = imp_sec.head(20)
axes[1, 1].barh(range(len(top_sec)), top_sec["importance"], color="steelblue")
axes[1, 1].set_yticks(range(len(top_sec)))
axes[1, 1].set_yticklabels(top_sec["feature"], fontsize=8)
axes[1, 1].invert_yaxis()
axes[1, 1].set_xlabel("Feature Importance (Gain)")
axes[1, 1].set_title("Top 20 Features")

plt.tight_layout()
plt.show()

In [ ]:
def run_subset_regression_sec(df_subset, label):
    """針對子資料集進行秒級預測的 XGBoost Regression 分析。"""
    TARGET_SEC = "reservation_seconds"
    LEAKAGE_SEC = ["reservation_hour", "reservation_half_hour"]

    df_clean = df_subset.drop(columns=LEAKAGE_SEC, errors="ignore")
    X_tr, X_te, y_tr, y_te = split_data(df_clean, target_col=TARGET_SEC)
    X_tr_proc, X_te_proc, _ = preprocess_for_tree(X_tr, X_te)

    mdl = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1,
    )
    mdl.fit(X_tr_proc, y_tr)
    y_pr = mdl.predict(X_te_proc)

    sub_rmse = mean_squared_error(y_te, y_pr)
    sub_mae = mean_absolute_error(y_te, y_pr)
    sub_r2 = r2_score(y_te, y_pr)

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Samples : {len(df_subset):,}  (Train {len(X_tr):,} / Test {len(X_te):,})")
    print(f"  RMSE : {sub_rmse:,.0f} sec  ({sub_rmse/60:.1f} min)")
    print(f"  MAE  : {sub_mae:,.0f} sec  ({sub_mae/60:.1f} min)")
    print(f"  R²   : {sub_r2:.4f}")

    residuals = y_te - y_pr

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"{label} — XGBoost (Seconds Regression)", fontsize=14, fontweight="bold")

    # (0,0) Actual vs Predicted
    axes[0, 0].scatter(y_te, y_pr, alpha=0.2, s=8, color="steelblue")
    lims = [min(y_te.min(), y_pr.min()), max(y_te.max(), y_pr.max())]
    axes[0, 0].plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
    axes[0, 0].set_xlabel("Actual (seconds)")
    axes[0, 0].set_ylabel("Predicted (seconds)")
    axes[0, 0].set_title("Actual vs Predicted")
    axes[0, 0].legend()

    # (0,1) Residual distribution
    axes[0, 1].hist(residuals / 60, bins=50, color="steelblue", edgecolor="white")
    axes[0, 1].axvline(0, color="red", linestyle="--")
    axes[0, 1].set_xlabel("Residual (minutes)")
    axes[0, 1].set_ylabel("Count")
    axes[0, 1].set_title(f"Residual Distribution (MAE={sub_mae/60:.1f} min)")

    # (1,0) Target distribution
    axes[1, 0].hist(df_subset[TARGET_SEC], bins=50, color="steelblue", edgecolor="white")
    axes[1, 0].set_xlabel("Seconds since midnight")
    axes[1, 0].set_ylabel("Count")
    axes[1, 0].set_title("Target Distribution")

    # (1,1) Feature importance top 15
    imp_df = pd.DataFrame({
        "feature": X_tr_proc.columns,
        "importance": mdl.feature_importances_
    }).sort_values("importance", ascending=False)
    top = imp_df.head(15)
    axes[1, 1].barh(range(len(top)), top["importance"], color="steelblue")
    axes[1, 1].set_yticks(range(len(top)))
    axes[1, 1].set_yticklabels(top["feature"], fontsize=8)
    axes[1, 1].invert_yaxis()
    axes[1, 1].set_xlabel("Feature Importance (Gain)")
    axes[1, 1].set_title("Top 15 Features")

    plt.tight_layout()
    plt.show()

    return {"label": label, "n": len(df_subset), "rmse_sec": sub_rmse, "mae_sec": sub_mae, "r2": sub_r2}

# --- Split & Run (三分法) ---
df_sec_lunch = df_sec[df_sec["reservation_hour"].between(11, 14)]
df_sec_afternoon = df_sec[df_sec["reservation_hour"].between(15, 17)]
df_sec_dinner = df_sec[df_sec["reservation_hour"].between(18, 23)]

print(f"午餐 Lunch      (11-14h): {len(df_sec_lunch):>6,} rows")
print(f"下午茶 Afternoon (15-17h): {len(df_sec_afternoon):>6,} rows")
print(f"晚餐 Dinner     (18-23h): {len(df_sec_dinner):>6,} rows")

results_sec = []
for subset, label in [
    (df_sec_lunch,     "午餐 Lunch (11-14h)"),
    (df_sec_afternoon, "下午茶 Afternoon Tea (15-17h)"),
    (df_sec_dinner,    "晚餐 Dinner (18-23h)"),
]:
    results_sec.append(run_subset_regression_sec(subset, label))

In [ ]:
# ============================================================
#  Comparison Summary — Seconds Regression (vs Full Dataset)
# ============================================================
print("="*70)
print("  COMPARISON SUMMARY — Seconds-Level Regression (vs Full Dataset)")
print("="*70)

full_result_sec = {
    "label": "整體 Full Dataset",
    "n": len(df_sec),
    "rmse_sec": rmse_sec,
    "mae_sec": mae_sec,
    "r2": r2_sec,
}
all_results_sec = [full_result_sec] + results_sec

comp_df_sec = pd.DataFrame(all_results_sec)
comp_df_sec["rmse_min"] = comp_df_sec["rmse_sec"] / 60
comp_df_sec["mae_min"] = comp_df_sec["mae_sec"] / 60

display(comp_df_sec[["label", "n", "rmse_min", "mae_min", "r2"]].rename(columns={
    "label": "Subset", "n": "Samples", "rmse_min": "RMSE (min)", "mae_min": "MAE (min)", "r2": "R²"
}).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
labels_sec = [r["label"] for r in all_results_sec]
colors_sec = ["gray", "steelblue", "coral", "seagreen"]
x_sec = range(len(labels_sec))

# RMSE (minutes)
axes[0].bar(x_sec, comp_df_sec["rmse_min"], color=colors_sec)
axes[0].set_xticks(list(x_sec))
axes[0].set_xticklabels(labels_sec, fontsize=9, rotation=15, ha="right")
axes[0].set_title("RMSE (minutes)")
axes[0].set_ylabel("Minutes")

# MAE (minutes)
axes[1].bar(x_sec, comp_df_sec["mae_min"], color=colors_sec)
axes[1].set_xticks(list(x_sec))
axes[1].set_xticklabels(labels_sec, fontsize=9, rotation=15, ha="right")
axes[1].set_title("MAE (minutes)")
axes[1].set_ylabel("Minutes")

# R²
axes[2].bar(x_sec, comp_df_sec["r2"], color=colors_sec)
axes[2].set_xticks(list(x_sec))
axes[2].set_xticklabels(labels_sec, fontsize=9, rotation=15, ha="right")
axes[2].set_title("R²")
axes[2].set_ylim(0, 1)

plt.suptitle("Full vs Subset — Seconds Regression (XGBoost Regressor)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 全模型 vs 三分模型：用餐時段預測分布比較

將三個子模型（午餐 / 下午茶 / 晚餐）的測試集預測合併成一條完整時段分布，
再與全模型的預測分布並列，**共用 x/y 軸尺度**，x 軸以一天 24 小時整點標示。

In [ ]:
# --- Full Model vs Combined Sub-Models: Distribution Comparison ---
# 為了拿到三個子模型的測試集預測，需要重新訓練（原 run_subset_analysis_hh 沒回傳預測）
import numpy as np

TARGET_HH = "reservation_half_hour"
LEAKAGE_HH = ["reservation_hour", "reservation_seconds"]

combined_actual_slots = []
combined_predicted_slots = []

for df_subset, label in [
    (df_hh_lunch,     "Lunch (11-14h)"),
    (df_hh_afternoon, "Afternoon Tea (14-18h)"),
    (df_hh_dinner,    "Dinner (18-23h)"),
]:
    df_clean = df_subset.drop(columns=LEAKAGE_HH, errors="ignore")
    X_tr, X_te, y_tr, y_te = split_data(df_clean, target_col=TARGET_HH)
    X_tr_proc, X_te_proc, _ = preprocess_for_tree(X_tr, X_te)

    sub_enc = LabelEncoder()
    y_tr_enc = sub_enc.fit_transform(y_tr)
    y_te_enc = sub_enc.transform(y_te)

    mdl = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, eval_metric="mlogloss",
    )
    mdl.fit(X_tr_proc, y_tr_enc)
    y_pr_enc = mdl.predict(X_te_proc)

    # 解碼回 0-47 半小時 slot 空間
    combined_actual_slots.extend(sub_enc.inverse_transform(y_te_enc))
    combined_predicted_slots.extend(sub_enc.inverse_transform(y_pr_enc))
    print(f"  {label}: test={len(y_te):,} samples")

combined_actual_slots = np.asarray(combined_actual_slots)
combined_predicted_slots = np.asarray(combined_predicted_slots)
print(f"\n合併後三分模型 test set 總筆數：{len(combined_actual_slots):,}")
print(f"全模型 test set 總筆數：{len(y_test_hh_slots):,}")

# --- Plot ---
bins = np.arange(18, 49)  # 只取 9:00 (slot 18) 之後的 half-hour slots
hour_ticks = np.arange(18, 49, 2)  # 從 9 點 (slot 18) 開始，每 2 個 slot = 1 小時
hour_labels = [str(h) for h in range(9, 25)]

# 計算共用 y 軸上限
# bins 已從 slot 18 開始，9 點前資料不會進入 histogram
y_max = 0
for arr in [y_test_hh_slots, y_pred_hh_slots, combined_actual_slots, combined_predicted_slots]:
    counts, _ = np.histogram(arr, bins=bins)
    y_max = max(y_max, counts.max())
y_max *= 1.05

fig, axes = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
fig.suptitle("Distribution Comparison: Full Model vs 3 Sub-Models (Half-Hour Granularity)",
             fontsize=14, fontweight="bold")

# Left: 全模型
axes[0].hist(y_test_hh_slots, bins=bins, alpha=0.5, label="Actual",
             color="steelblue", edgecolor="white", align="left")
axes[0].hist(y_pred_hh_slots, bins=bins, alpha=0.5, label="Predicted",
             color="coral", edgecolor="white", align="left")
axes[0].set_title(f"全模型 Full Model (n={len(y_test_hh_slots):,})", fontsize=12)
axes[0].set_ylabel("Count")
axes[0].legend()

# Right: 三分模型合併
axes[1].hist(combined_actual_slots, bins=bins, alpha=0.5, label="Actual",
             color="steelblue", edgecolor="white", align="left")
axes[1].hist(combined_predicted_slots, bins=bins, alpha=0.5, label="Predicted",
             color="coral", edgecolor="white", align="left")
axes[1].set_title(f"三分模型合併 Sub-Models Combined (n={len(combined_actual_slots):,})", fontsize=12)
axes[1].legend()

for ax in axes:
    ax.set_xticks(hour_ticks)
    ax.set_xticklabels(hour_labels)
    ax.set_xlim(18, 48)  # 隱藏 9 點以前的資料
    ax.set_ylim(0, y_max)
    ax.set_xlabel("Hour of Day")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 改善方向 C：序數化目標 (Ordinal Regression)

前面的 classifier 把 48 個半小時 slot 視為「48 個彼此獨立的標籤」，導致：
- 「預測 12:00 結果是 12:30」與「預測 12:00 結果是 22:00」被視為**同等錯誤**
- 模型沒有誘因把預測攤開到鄰近時段，反而傾向塌縮到單一尖峰

**做法**：把 `reservation_half_hour` (0-47) 當成**連續數值**，用 `XGBRegressor` 配
`objective="reg:absoluteerror"` (MAE) 訓練，預測完 round 回最近的 slot。

- 損失函數懲罰「離正確時段的距離」，鄰近錯誤 = 小錯
- 模型有動機讓預測分布貼合真實分布，不再塌縮
- 評估指標：Accuracy（精準命中）、MAE (slots)、±1 slot / ±2 slots 命中率

訓練設定：全資料 + 三個子資料（午餐 / 下午茶 / 晚餐）各訓練一個 XGBRegressor。
最後與真實分布並列比較，**只顯示 9:00 之後**。

In [ ]:
# === 改善方向 C：序數化目標 (Ordinal Regression) ===
from xgboost import XGBRegressor

TARGET_HH = "reservation_half_hour"
LEAKAGE_HH = ["reservation_hour", "reservation_seconds"]


def train_predict_regression(df_subset, label):
    """訓練序數化 regression 模型，回傳 (y_true_slots, y_pred_slots)。"""
    df_clean = df_subset.drop(columns=LEAKAGE_HH, errors="ignore")
    X_tr, X_te, y_tr, y_te = split_data(df_clean, target_col=TARGET_HH)
    X_tr_proc, X_te_proc, _ = preprocess_for_tree(X_tr, X_te)

    mdl = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        objective="reg:absoluteerror",   # MAE：鄰近時段錯誤 = 小錯
        random_state=42, n_jobs=-1,
    )
    mdl.fit(X_tr_proc, y_tr)
    y_pr_raw = mdl.predict(X_te_proc)
    y_pr = np.clip(np.round(y_pr_raw).astype(int), 0, 47)
    y_te_arr = y_te.values.astype(int)

    acc      = (y_pr == y_te_arr).mean()
    mae      = np.abs(y_pr - y_te_arr).mean()
    within_1 = (np.abs(y_pr - y_te_arr) <= 1).mean()   # ±30 min
    within_2 = (np.abs(y_pr - y_te_arr) <= 2).mean()   # ±1 hour

    print(f"  {label}:")
    print(f"    test n         = {len(y_te_arr):,}")
    print(f"    Accuracy       = {acc:.4f}")
    print(f"    MAE (slots)    = {mae:.2f}   (≈ {mae*30:.0f} min)")
    print(f"    Acc within ±1  = {within_1:.4f}   (容差 ±30 min)")
    print(f"    Acc within ±2  = {within_2:.4f}   (容差 ±1 hr)")
    return y_te_arr, y_pr


# --- Full model regression ---
print("=" * 60)
print("Full Model (Regression)")
print("=" * 60)
y_true_full_reg, y_pred_full_reg = train_predict_regression(df_hh, "Full")

# --- 3 sub-models regression ---
print("\n" + "=" * 60)
print("Sub-Models (Regression)")
print("=" * 60)
combined_true_reg = []
combined_pred_reg = []
for df_subset, label in [
    (df_hh_lunch,     "Lunch (11-14h)"),
    (df_hh_afternoon, "Afternoon (15-17h)"),
    (df_hh_dinner,    "Dinner (18-23h)"),
]:
    y_te_arr, y_pr = train_predict_regression(df_subset, label)
    combined_true_reg.extend(y_te_arr)
    combined_pred_reg.extend(y_pr)

combined_true_reg = np.asarray(combined_true_reg)
combined_pred_reg = np.asarray(combined_pred_reg)
print(f"\n合併三分模型 test set 總筆數：{len(combined_true_reg):,}")

# --- Plot: side-by-side, ≥ 9:00 only ---
bins = np.arange(18, 49)                       # slot 18 (9:00) ~ slot 47 (23:30)
hour_ticks = np.arange(18, 49, 2)              # 每 2 個 slot = 1 小時
hour_labels = [str(h) for h in range(9, 25)]

y_max = 0
for arr in [y_true_full_reg, y_pred_full_reg, combined_true_reg, combined_pred_reg]:
    counts, _ = np.histogram(arr, bins=bins)
    y_max = max(y_max, counts.max())
y_max *= 1.05

fig, axes = plt.subplots(1, 2, figsize=(20, 6), sharey=True)
fig.suptitle("改善 C — 序數化 Regression：Full Model vs 3 Sub-Models (≥ 9:00)",
             fontsize=14, fontweight="bold")

axes[0].hist(y_true_full_reg, bins=bins, alpha=0.5, label="Actual",
             color="steelblue", edgecolor="white", align="left")
axes[0].hist(y_pred_full_reg, bins=bins, alpha=0.5, label="Predicted",
             color="coral", edgecolor="white", align="left")
axes[0].set_title(f"全模型 Full Model (n={len(y_true_full_reg):,})", fontsize=12)
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(combined_true_reg, bins=bins, alpha=0.5, label="Actual",
             color="steelblue", edgecolor="white", align="left")
axes[1].hist(combined_pred_reg, bins=bins, alpha=0.5, label="Predicted",
             color="coral", edgecolor="white", align="left")
axes[1].set_title(f"三分模型合併 Sub-Models Combined (n={len(combined_true_reg):,})", fontsize=12)
axes[1].legend()

for ax in axes:
    ax.set_xticks(hour_ticks)
    ax.set_xticklabels(hour_labels)
    ax.set_xlim(18, 48)
    ax.set_ylim(0, y_max)
    ax.set_xlabel("Hour of Day")
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


### 評量指標比較：全模型 vs 三分模型 (Regression)

比較四個指標：
- **Accuracy**：精準命中正確半小時 slot 的比例
- **MAE (slots)**：平均離正確 slot 多遠（單位 = 半小時）
- **Acc ±1 slot**：容差 ±30 分鐘的命中率
- **Acc ±2 slot**：容差 ±1 小時的命中率

直接使用上一個 cell 留下的 `y_true_full_reg / y_pred_full_reg` 與
`combined_true_reg / combined_pred_reg`，**不需重新訓練**。

In [ ]:
# === Regression Metrics: Full Model vs 3-Sub Combined ===

def compute_reg_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    diff = np.abs(y_true - y_pred)
    return {
        "Accuracy":    (y_true == y_pred).mean(),
        "MAE (slots)": diff.mean(),
        "Acc +/-1":    (diff <= 1).mean(),
        "Acc +/-2":    (diff <= 2).mean(),
    }

m_full = compute_reg_metrics(y_true_full_reg, y_pred_full_reg)
m_sub  = compute_reg_metrics(combined_true_reg, combined_pred_reg)

# Print summary table
print(f"{'Metric':<15} {'Full Model':>12} {'3-Sub Combined':>16} {'Δ':>10}")
print("-" * 56)
for k in m_full:
    delta = m_sub[k] - m_full[k]
    sign = "+" if delta >= 0 else ""
    print(f"{k:<15} {m_full[k]:>12.4f} {m_sub[k]:>16.4f} {sign}{delta:>9.4f}")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Regression Metrics: Full Model vs 3-Sub Combined",
             fontsize=14, fontweight="bold")

# Left: 0-1 ratio metrics (Accuracy / within ±1 / within ±2)
ratio_metrics = ["Accuracy", "Acc +/-1", "Acc +/-2"]
x = np.arange(len(ratio_metrics))
width = 0.35
full_vals = [m_full[k] for k in ratio_metrics]
sub_vals  = [m_sub[k]  for k in ratio_metrics]

b1 = axes[0].bar(x - width/2, full_vals, width,
                 label="Full Model", color="steelblue")
b2 = axes[0].bar(x + width/2, sub_vals,  width,
                 label="3-Sub Combined", color="coral")
axes[0].set_xticks(x)
axes[0].set_xticklabels(["Accuracy", "Acc +/-1 slot\n(+/-30 min)", "Acc +/-2 slot\n(+/-1 hr)"])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Ratio")
axes[0].set_title("Accuracy-style Metrics")
axes[0].legend(loc="lower right")
axes[0].grid(axis="y", alpha=0.3)
for b in list(b1) + list(b2):
    h = b.get_height()
    axes[0].text(b.get_x() + b.get_width()/2, h + 0.01, f"{h:.3f}",
                 ha="center", va="bottom", fontsize=9)

# Right: MAE in slot units (different scale)
labels_r = ["Full Model", "3-Sub Combined"]
mae_vals = [m_full["MAE (slots)"], m_sub["MAE (slots)"]]
b3 = axes[1].bar(labels_r, mae_vals,
                 color=["steelblue", "coral"], width=0.5)
axes[1].set_ylabel("MAE (half-hour slots)")
axes[1].set_title("Mean Absolute Error (smaller = better)")
axes[1].set_ylim(0, max(mae_vals) * 1.30)
axes[1].grid(axis="y", alpha=0.3)
for b in b3:
    h = b.get_height()
    axes[1].text(b.get_x() + b.get_width()/2, h + 0.05,
                 f"{h:.2f}\n(~{h*30:.0f} min)",
                 ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()
